# 02 — Thu thập và chuẩn hóa các nguồn dữ liệu ngoài

Bộ Expedia chỉ có thông tin chuyến bay của **năm 2022**. Để phân tích sâu hơn và dự báo cho 2023–2027,
ta tích hợp thêm 7 nguồn:

| # | Nguồn | Mức chi tiết | Dùng để |
|---|---|---|---|
| 1 | **OpenFlights** — danh mục sân bay | sân bay | thành phố, bang, tọa độ, múi giờ |
| 2 | Thư viện **holidays** — ngày lễ liên bang Mỹ | ngày | `is_holiday`, số ngày tới ngày lễ gần nhất |
| 3 | **Open-Meteo** — thời tiết lịch sử | sân bay × ngày | nhiệt độ, mưa, tuyết, gió |
| 4 | **EIA** — giá nhiên liệu bay (Jet Fuel, U.S. Gulf Coast) | ngày, tháng | chi phí lớn nhất của hãng bay |
| 5 | **BLS** — CPI vé máy bay & CPI chung (lạm phát) | tháng, 2005 → 08/2026 | xu hướng giá nhiều năm, lạm phát |
| 6 | **DOT Consumer Airfare Report (Table 1a)** | tuyến × quý, 1993 → Q1/2024 | **giá vé thực tế 2023** để kiểm chứng |
| 7 | **BTS Average Domestic Itinerary Fare** | sân bay × quý, 1993 → Q3/2024 | giá thực tế theo sân bay |

Kết quả ghi vào `E:/flight-data/external/clean/` (CSV nhỏ, dùng cho Spark join và mô hình dự báo).

In [1]:
import sys, os, re, io, glob, time
sys.path.insert(0, os.path.abspath(".."))
from common import *
import pandas as pd, numpy as np, requests

CLEAN = f"{EXTERNAL}/clean"
os.makedirs(CLEAN, exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/128 Safari/537.36"}

# 16 sân bay có trong bộ Expedia (xem notebook 01 / xem_du_lieu.ipynb)
AIRPORTS = ["ATL", "BOS", "CLT", "DEN", "DFW", "DTW", "EWR", "IAD",
            "JFK", "LAX", "LGA", "MIA", "OAK", "ORD", "PHL", "SFO"]

## 1. Sân bay — OpenFlights

In [2]:
ap_cols = ["id", "name", "city", "country", "iata", "icao", "lat", "lon",
           "alt", "tz_offset", "dst", "tzdb", "type", "source"]
ap = pd.read_csv("https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat",
                 header=None, names=ap_cols, na_values="\\N")
airports = (ap[ap["iata"].isin(AIRPORTS)][["iata", "name", "city", "lat", "lon", "alt", "tzdb"]]
            .sort_values("iata").reset_index(drop=True))
# Bờ Đông / Trung tâm / Bờ Tây theo múi giờ
airports["region"] = airports["tzdb"].map({"America/New_York": "East", "America/Detroit": "East",
                                           "America/Chicago": "Central", "America/Denver": "Mountain",
                                           "America/Los_Angeles": "West"})
airports.to_csv(f"{CLEAN}/airports.csv", index=False)
airports

,iata,name,city,lat,lon,alt,tzdb,region
0,ATL,Hartsfield Jackson Atlanta International Airport,Atlanta,33.636700,-84.428101,1026,America/New_York,East
1,BOS,General Edward Lawrence Logan International Ai...,Boston,42.364300,-71.005203,20,America/New_York,East
2,CLT,Charlotte Douglas International Airport,Charlotte,35.214001,-80.943100,748,America/New_York,East
3,DEN,Denver International Airport,Denver,39.861698,-104.672997,5431,America/Denver,Mountain
4,DFW,Dallas Fort Worth International Airport,Dallas-Fort Worth,32.896801,-97.038002,607,America/Chicago,Central
5,DTW,Detroit Metropolitan Wayne County Airport,Detroit,42.212399,-83.353401,645,America/New_York,East
6,EWR,Newark Liberty International Airport,Newark,40.692501,-74.168701,18,America/New_York,East
7,IAD,Washington Dulles International Airport,Washington,38.944500,-77.455803,312,America/New_York,East
8,JFK,John F Kennedy International Airport,New York,40.639801,-73.778900,13,America/New_York,East
9,LAX,Los Angeles International Airport,Los Angeles,33.942501,-118.407997,125,America/Los_Angeles,West


## 2. Ngày lễ liên bang Mỹ — thư viện `holidays` (2022 → 2027)

In [3]:
import holidays as holidays_lib
us = holidays_lib.US(years=range(2022, 2028))
hol = (pd.DataFrame({"date": pd.to_datetime(list(us.keys())), "holiday_name": list(us.values())})
       .sort_values("date").reset_index(drop=True))
hol.to_csv(f"{CLEAN}/holidays.csv", index=False)
print(len(hol), "ngày lễ;", "trong khoảng bay của dữ liệu 2022:")
hol[(hol.date >= "2022-04-01") & (hol.date <= "2022-11-30")]

75 ngày lễ; trong khoảng bay của dữ liệu 2022:


,date,holiday_name
3,2022-05-30,Memorial Day
4,2022-06-19,Juneteenth National Independence Day
5,2022-06-20,Juneteenth National Independence Day (observed)
6,2022-07-04,Independence Day
7,2022-09-05,Labor Day
8,2022-10-10,Columbus Day
9,2022-11-11,Veterans Day
10,2022-11-24,Thanksgiving Day


## 3. Thời tiết lịch sử — Open-Meteo (16 sân bay, 01/04 → 30/11/2022, theo ngày)

In [4]:
frames = []
for _, a in airports.iterrows():
    for attempt in range(5):
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
            "latitude": a.lat, "longitude": a.lon, "start_date": "2022-04-01", "end_date": "2022-11-30",
            "daily": "temperature_2m_mean,temperature_2m_max,precipitation_sum,snowfall_sum,wind_speed_10m_max",
            "timezone": a.tzdb})
        if r.status_code == 200:
            break
        time.sleep(10 * (attempt + 1))      # API giới hạn tần suất → chờ rồi thử lại
    r.raise_for_status()
    d = r.json()["daily"]
    frames.append(pd.DataFrame({
        "airport": a.iata, "date": pd.to_datetime(d["time"]),
        "temp_mean": d["temperature_2m_mean"], "temp_max": d["temperature_2m_max"],
        "precip_mm": d["precipitation_sum"], "snow_cm": d["snowfall_sum"], "wind_max": d["wind_speed_10m_max"]}))
    time.sleep(1)
weather = pd.concat(frames, ignore_index=True)
weather["bad_weather"] = ((weather.precip_mm >= 10) | (weather.snow_cm >= 1) | (weather.wind_max >= 40)).astype(int)
weather.to_csv(f"{CLEAN}/weather_daily.csv", index=False)
print(weather.shape)
weather.describe().round(1)

(3904, 8)


,date,temp_mean,temp_max,precip_mm,snow_cm,wind_max,bad_weather
count,3904,3904.0,3904.0,3904.0,3904.0,3904.0,3904.0
mean,2022-07-31 12:00:00,18.4,23.6,3.0,0.0,19.9,0.1
min,2022-04-01 00:00:00,-8.4,-4.7,0.0,0.0,6.5,0.0
25%,2022-05-31 18:00:00,13.7,18.7,0.0,0.0,15.2,0.0
50%,2022-07-31 12:00:00,19.2,24.1,0.0,0.0,19.1,0.0
75%,2022-09-30 06:00:00,24.1,29.4,1.9,0.0,23.8,0.0
max,2022-11-30 00:00:00,36.0,43.0,151.6,7.4,49.6,1.0
std,NaN,7.0,7.5,8.0,0.2,6.5,0.3


## 4. Giá nhiên liệu bay — EIA (Kerosene-Type Jet Fuel, U.S. Gulf Coast, USD/gallon)

In [5]:
def eia_series(code, freq):
    path = f"{EXTERNAL}/eia_{code}_{freq}.xls"
    if not os.path.exists(path):
        r = requests.get(f"https://www.eia.gov/dnav/pet/hist_xls/{code}{freq}.xls", headers=UA, timeout=120)
        r.raise_for_status()
        open(path, "wb").write(r.content)
    s = pd.read_excel(path, sheet_name="Data 1", skiprows=2, engine="xlrd")
    s.columns = ["date", "jet_fuel_usd_gal"]
    return s.dropna()

fuel_d = eia_series("EER_EPJK_PF4_RGC_DPG", "d")
fuel_m = eia_series("EER_EPJK_PF4_RGC_DPG", "m")
fuel_m["date"] = fuel_m["date"].dt.to_period("M").dt.to_timestamp()
fuel_d.to_csv(f"{CLEAN}/jet_fuel_daily.csv", index=False)
print("Daily:", fuel_d.date.min().date(), "→", fuel_d.date.max().date(),
      "| Monthly:", fuel_m.date.min().date(), "→", fuel_m.date.max().date())

Daily: 1990-04-02 → 2026-09-29 | Monthly: 1990-04-01 → 2026-08-01


## 5. Lạm phát — BLS CPI (không điều chỉnh mùa vụ, 1982–84 = 100)

- `CUUR0000SETG01`: CPI **vé máy bay** (Airline fares)
- `CUUR0000SA0`: CPI **chung** (All items) → dùng để tính lạm phát

In [6]:
rows = []
for f in sorted(glob.glob(f"{EXTERNAL}/bls_*.json")):
    import json
    for s in json.load(open(f))["Results"]["series"]:
        for x in s["data"]:
            if x["period"].startswith("M") and x["period"] != "M13":
                value = float(x["value"]) if x["value"] not in ("-", "") else np.nan
                rows.append((s["seriesID"], f"{x['year']}-{x['period'][1:]}-01", value))
cpi = (pd.DataFrame(rows, columns=["series", "date", "value"]).drop_duplicates()
       .pivot(index="date", columns="series", values="value")
       .rename(columns={"CUUR0000SETG01": "cpi_airfare", "CUUR0000SA0": "cpi_all"}))
cpi.index = pd.to_datetime(cpi.index)
cpi = cpi.sort_index()
missing = cpi[cpi.isna().any(axis=1)]
print("Tháng BLS không công bố số liệu (nội suy tuyến tính):", [d.strftime("%Y-%m") for d in missing.index])
cpi = cpi.interpolate()
cpi["inflation_yoy"] = cpi["cpi_all"].pct_change(12) * 100          # lạm phát so với cùng kỳ (%)
cpi["airfare_yoy"] = cpi["cpi_airfare"].pct_change(12) * 100

macro = cpi.join(fuel_m.set_index("date"), how="left").reset_index().rename(columns={"index": "date"})
macro.to_csv(f"{CLEAN}/macro_monthly.csv", index=False)
print("Macro theo tháng:", macro.date.min().date(), "→", macro.date.max().date(), macro.shape)
macro.tail(10).round(2)

Tháng BLS không công bố số liệu (nội suy tuyến tính): ['2025-10']
Macro theo tháng: 2005-01-01 → 2026-08-01 (260, 6)


C:\Users\HP\AppData\Local\Temp\ipykernel_26092\1266045176.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  macro.tail(10).round(2)


,date,cpi_all,cpi_airfare,inflation_yoy,airfare_yoy,jet_fuel_usd_gal
250,2025-11-01,324.12,254.92,2.74,-5.35,2.26
251,2025-12-01,324.05,253.71,2.68,-3.37,1.96
252,2026-01-01,325.25,271.19,2.39,2.23,2.03
253,2026-02-01,326.79,280.62,2.41,7.05,2.26
254,2026-03-01,330.21,290.23,3.26,14.89,3.70
255,2026-04-01,333.02,308.53,3.81,20.71,3.93
256,2026-05-01,335.12,329.82,4.25,26.70,3.94
257,2026-06-01,333.95,323.76,3.53,26.54,3.04
258,2026-07-01,333.92,311.18,3.36,25.55,3.40
259,2026-08-01,334.98,309.74,3.40,23.41,3.72


## 6. Giá vé thực tế theo tuyến — DOT Consumer Airfare Report, Table 1a

Trang gốc `data.transportation.gov` chặn truy cập từ Việt Nam (HTTP 403), nên dùng bản sao trên Kaggle
(*bhavikjikadara/us-airline-flight-routes-and-fares-1993-2024*). Đây là giá trung bình **một chiều** theo
**cặp sân bay** (không phân biệt chiều đi/về), mỗi quý.

In [7]:
dot = pd.read_csv(glob.glob(f"{EXTERNAL}/bhavikjikadara*/*.csv")[0], low_memory=False)
dot = dot[dot.airport_1.isin(AIRPORTS) & dot.airport_2.isin(AIRPORTS)].copy()
dot["pair"] = [ "-".join(sorted(p)) for p in zip(dot.airport_1, dot.airport_2)]
# Một cặp sân bay có thể xuất hiện nhiều dòng trong cùng quý → lấy trung bình có trọng số theo số hành khách
dot = dot.rename(columns={"Year": "year"})
dot["w"] = dot["passengers"].clip(lower=1)
dot["fare_x_w"] = dot["fare"] * dot["w"]
dot_q = (dot.groupby(["pair", "year", "quarter"], as_index=False)
         .agg(fare_x_w=("fare_x_w", "sum"), w=("w", "sum"), passengers=("passengers", "sum"),
              nsmiles=("nsmiles", "mean"), fare_low=("fare_low", "mean")))
dot_q["fare"] = dot_q.pop("fare_x_w") / dot_q.pop("w")
dot_q.to_csv(f"{CLEAN}/dot_route_quarter.csv", index=False)
print("Số cặp sân bay:", dot_q.pair.nunique(), "| năm:", dot_q.year.min(), "→", dot_q.year.max())
dot_q.groupby("year").agg(so_tuyen=("pair", "nunique"), gia_tb=("fare", "mean")).tail(8).round(1)

Số cặp sân bay: 109 | năm: 1993 → 2024


,so_tuyen,gia_tb
year,,
2017,102,225.6
2018,102,232.1
2019,102,242.2
2020,102,194.8
2021,102,205.3
2022,102,253.6
2023,102,252.3
2024,102,257.4


## 7. Giá vé trung bình theo sân bay — BTS (1993 → Q3/2024)

File `.xls` thực chất là Excel XML 2003 và bị cắt cụt ở cuối, nên đọc bằng biểu thức chính quy.

In [8]:
def read_bts(path):
    text = open(path, encoding="utf-8-sig", errors="ignore").read()
    out = []
    for row in re.findall(r"<Row.*?</Row>", text, flags=re.S):
        cells = re.findall(r"<Data[^>]*>(.*?)</Data>", row, flags=re.S)
        if len(cells) >= 6 and cells[1] in AIRPORTS:
            out.append((cells[1], float(cells[5])))
    return out

bts_rows = []
for f in glob.glob(f"{EXTERNAL}/samithsachidanandan*/AverageFare_Q*_*.xls"):
    q, y = re.search(r"Q(\d)_(\d{4})", f).groups()
    bts_rows += [(code, int(y), int(q), fare) for code, fare in read_bts(f)]
bts = pd.DataFrame(bts_rows, columns=["airport", "year", "quarter", "avg_fare"]).sort_values(["airport", "year", "quarter"])
bts.to_csv(f"{CLEAN}/bts_airport_quarter.csv", index=False)
print(bts.shape, "| năm:", bts.year.min(), "→", bts.year.max())
bts.pivot_table(index="year", columns="airport", values="avg_fare").tail(6).round(0)

(2032, 4) | năm: 1993 → 2024


airport,ATL,BOS,CLT,DEN,DFW,DTW,EWR,IAD,JFK,LAX,LGA,MIA,OAK,ORD,PHL,SFO
year,,,,,,,,,,,,,,,,
2019,371.0,337.0,413.0,301.0,403.0,378.0,407.0,456.0,402.0,356.0,334.0,335.0,295.0,345.0,371.0,397.0
2020,283.0,264.0,319.0,242.0,305.0,294.0,280.0,345.0,315.0,281.0,250.0,250.0,232.0,254.0,268.0,319.0
2021,297.0,315.0,330.0,265.0,313.0,323.0,303.0,386.0,327.0,310.0,275.0,247.0,263.0,279.0,298.0,373.0
2022,390.0,383.0,440.0,339.0,417.0,425.0,400.0,489.0,413.0,411.0,315.0,323.0,286.0,371.0,394.0,449.0
2023,397.0,384.0,438.0,347.0,422.0,427.0,406.0,488.0,421.0,405.0,328.0,344.0,303.0,384.0,401.0,443.0
2024,406.0,370.0,426.0,345.0,417.0,425.0,419.0,484.0,424.0,400.0,322.0,334.0,313.0,372.0,392.0,447.0


## Tóm tắt các bảng đã chuẩn hóa

In [9]:
summary = pd.DataFrame([(os.path.basename(f), len(pd.read_csv(f)), round(os.path.getsize(f) / 1e3, 1))
                        for f in sorted(glob.glob(f"{CLEAN}/*.csv"))],
                       columns=["bảng", "số dòng", "KB"])
summary.to_csv(f"{EXPORTS}/02_external_sources_summary.csv", index=False)
summary

,bảng,số dòng,KB
0,airports.csv,16,1.7
1,bts_airport_quarter.csv,2032,61.8
2,dot_route_quarter.csv,12175,507.7
3,holidays.csv,75,2.4
4,jet_fuel_daily.csv,8046,143.9
5,macro_monthly.csv,260,18.1
6,weather_daily.csv,3904,159.8
